In [ ]:
import os
from openai import OpenAI
api_key=os.environ.get("Agnes_Api_Key")
client=OpenAI(api_key=api_key,base_url="https://apihub.agnes-ai.com/v1")
model="agnes-2.0-flash"
print("="*40);print("1.零样本 Prompt")
response=client.chat.completions.create(#调用客户端的chat.completions.create方法,向模型发送一个对话补齐请求
    model=model,#模型名称
    messages=[{"role":"user","content":"将下面句子翻译成英文:今天天气很好"}],#角色是role=user,发送content
   temperature=0.7#控制生成随机性的参数。值越大:输出越多样、有创造性。有可能输出is nice day/Today's weather的同义表达
)
print("结果:",response.choices[0].message.content)
print("="*40);print("2.少样本 Prompt")
response=client.chat.completions.create(
    model=model,temperature=0.7,
    messages=[{"role":"user","content":"例子:苹果->apple\n橘子->orange\n猫->?"}] )
print("结果:",response.choices[0].message.content)
print("="*40);print("3.思维链Prompt")
cot_prompt = """
问题:笼子里有鸡和兔,共有10个头,36条腿。问鸡和兔各几只?
请一步步推理,最后给出答案。不用latex数学公式用+-*/等普通格式表示公式
"""
response=client.chat.completions.create(
    model=model,temperature=0.7,
    messages=[{"role": "user", "content": cot_prompt}]
)#把隐式推理变成显示步骤,减少跳跃性错误。
print("结果:",response.choices[0].message.content)
print("="*40);print("4.格式约束Prompt")
response=client.chat.completions.create(
    model=model,temperature=0.7,
    messages=[{"role":"user","content":"以json方式输出int [2][2]和double和自定义类型Date数据"}])
print("结果:",response.choices[0].message.content)

1.零样本 Prompt
结果: The weather is very nice today.
2.少样本 Prompt
结果: cat
3.思维链Prompt
结果: 我们可以通过假设法来一步步推理这个问题：

**第一步：假设全是鸡**
如果笼子里10个头全部都是鸡，那么腿的总数应该是：
$10 \times 2 = 20$ 条腿

**第二步：计算腿数差额**
题目中实际有36条腿，而我们假设全是鸡时只有20条腿。多出来的腿数是：
$36 - 20 = 16$ 条腿

**第三步：分析每只兔子比鸡多几条腿**
一只兔子有4条腿，一只鸡有2条腿。每把一只鸡换成兔子，腿的数量就会增加：
$4 - 2 = 2$ 条腿

**第四步：计算兔子的数量**
多出来的16条腿是因为我们把鸡换成了兔子。既然每换一只兔子多出2条腿，那么兔子的数量就是：
$16 / 2 = 8$ 只

**第五步：计算鸡的数量**
总共有10个头（也就是10只动物），减去兔子的数量，剩下的就是鸡的数量：
$10 - 8 = 2$ 只

**第六步：验证答案**
- 鸡2只，腿数：$2 \times 2 = 4$ 条
- 兔8只，腿数：$8 \times 4 = 32$ 条
- 总腿数：$4 + 32 = 36$ 条
- 总头数：$2 + 8 = 10$ 个
答案符合题意。

**最终答案：**
鸡有2只，兔有8只。
4.格式约束Prompt
结果: ```json
{
  "intArray": [
    [1, 2],
    [3, 4]
  ],
  "doubleValue": 3.14159,
  "dateObject": {
    "year": 2023,
    "month": 10,
    "day": 5
  }
}
```


In [2]:
import os
#Langchain核心组件
#PromptTemplate是一个带变量的字符串模板,可在运行时替代变量
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
llm=ChatOpenAI(
    api_key=os.environ.get("Agnes_Api_Key"),
    base_url="https://apihub.agnes-ai.com/v1",
    model="agnes-2.0-flash",
    temperature=0.7
)
template=PromptTemplate(
    input_variables=["topic","theme","lines"],
    template="请写一首关于{topic}的{theme},不超过{lines}行"
)
val1=template.format(topic="春天",theme="短诗",lines=4)
result=llm.invoke(val1)
print(result.content)

嫩芽破土迎晨曦，
细雨无声润花枝。
燕子归来寻旧巷，
春风拂面暖如诗。


In [ ]:
import os
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

# 配置 LLM
llm = ChatOpenAI(
    api_key=os.environ.get("AGNES_API_KEY"),
    base_url="https://apihub.agnes-ai.com/v1",
    model="agnes-2.0-flash",
    temperature=0.7
)

#定义多个模板
templates={
    "问答":PromptTemplate.from_template("请回答一下问题:{question}"),#from_template静态写法,穿字符串
    "翻译":PromptTemplate.from_template("请将以下文本翻译成:{language}:{text}"),
    "总结":PromptTemplate.from_template("请用一句话总结以下内容:{content}")
}

# 构建链（共享同一个 LLM）
"""
def chain(input_text):
    llm_result = llm.invoke(input_text)   # 先跑模型
    return StrOutputParser().invoke(llm_result)  # 再解析
    等价于下面的chain,而StrOutputParser()只取出里面真正的回答文本，丢弃对象属性、元数据。最终返回普通 str 字符串，方便后续处理
"""
chain = llm | StrOutputParser() #将llm给StrOutputParser

# 测试
print("=== 问答 ===")
result1 = (templates["问答"] | chain).invoke({"question": "什么是LangChain?"})
print(result1)
""" 等价于上面管道符
prompt1=templates["问答"].format(question="什么是LangChain?")
result1=chain.invoke(prompt1)
print(result1)
"""
lang="Yesterday is history, tomorrow is a mystery, but today is a gift,that's why it's called the present."
print("\n=== 翻译 ===")
result2=(templates["翻译"]|chain).invoke({"language":"中文","text":lang})
print(result2)
result2=(templates["翻译"]|chain).invoke({"language":"俄语","text":lang})
print(result2)
print("\n=== 总结 ===")
result3=(templates["总结"]|chain).invoke({"content":"LangChain 是一个用于构建大语言模型应用的框架。它提供了链式调用、提示词模板、向量数据库集成等功能，大大简化了 AI 应用开发的复杂度。"})
print(result3)

=== 问答 ===
LangChain 是一个用于构建基于大语言模型（LLM）的应用程序的开源框架。它旨在简化将 LLM 与外部数据源、其他应用程序以及人类交互相结合的过程。

以下是 LangChain 的核心特点和功能：

1. **组件化设计**：提供了一系列模块化组件，如链（Chains）、代理（Agents）、记忆（Memory）和文档加载器（Document Loaders），方便开发者快速组装复杂应用。
2. **上下文管理**：帮助处理长文本输入，通过检索增强生成（RAG）等技术，让 LLM 能够利用私有或实时数据。
3. **多模型支持**：兼容多种主流大语言模型提供商（如 OpenAI、Anthropic、Hugging Face 等），便于切换和集成。
4. **智能代理（Agents）**：允许 LLM 自主决定调用工具（如搜索、计算器、数据库查询）来完成任务，而不仅仅是生成文本。
5. **开发效率**：减少从零开始构建 LLM 应用所需的代码量，加速原型开发和生产部署。

简而言之，LangChain 是连接大语言模型与现实世界数据和操作之间的桥梁，帮助开发者更高效地创建智能化、上下文感知的 AI 应用。

=== 翻译 ===
昨天已成历史，明天尚是谜团，而今天则是上天的馈赠，正因如此，它才被称为“礼物”（present）。
Вчера — это история, завтра — тайна, но сегодня — подарок, именно поэтому оно называется «настоящее».

=== 总结 ===
LangChain 是一个通过提供链式调用、提示词模板及向量数据库集成等功能，从而简化大语言模型应用开发复杂度的框架。


In [4]:
#创建文档
sample_doc="""
AI Agent（Artificial Intelligence Agent） 称为智能体，本质是自动执行任务的程序，核心在于让模型不只回答问题，而是按步骤完成动作。
AI Agent（人工智能代理） 是一个能够感知环境、进行决策并执行行动，以达成特定目标的智能软件实体，它不仅仅是回答问题的聊天机器人，更是能够动手做事的智能执行者。
Agent = LLM (大脑) + Planning (规划) + Tool use (执行) + Memory (记忆)。
学习 Agent 需要思维转变： 从对话框问答进化为目标驱动的任务执行。
"""
#保存为txt文件
with open("test_doc.txt","w",encoding="utf-8") as f:
    f.write(sample_doc)
print("测试文档已创建")

测试文档已创建


In [ ]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
#配置LLM
llm=ChatOpenAI(api_key=os.environ.get("AGNES_API_KEY"),base_url="https://apihub.agnes-ai.com/v1",model="agnes-2.0-flash",temperature=0)
#1.加载文档
loader=TextLoader("test_doc.txt",encoding="utf-8")#创建加载器实例，只是记录文件路径
documents=loader.load()#load 打开文件,读取内容,包装成document对象,返回文档列表
#2.分割文本
text_splitter=RecursiveCharacterTextSplitter(chunk_size=200,chunk_overlap=80)#块大小200,块重叠20。递归分割,优先按行、段落、句号、逗号拆分,保持语义完整
chunks=text_splitter.split_documents(documents)#split输入原始文档,输出切分后的小段文本块列表chunks,每一块都是独立document
print(f"分割成{len(chunks)}个文档块")
#3.本地嵌入模型+向量库搭建
embeddings=HuggingFaceBgeEmbeddings(#Embedding将人类文字转换为计算机可计算的浮点向量,语义相近的文本向量距离更近,用于相似度匹配。
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)#使用开源语言MiniLM模型,支持中英文,轻量可在cpu运行
print("嵌入模型加载完成")
vector_store=Chroma.from_documents(documents=chunks,embedding=embeddings)
#Chroma.from_documents一次性完成两件事。1.遍历所有文本块,调用embedding模型生成每段文本向量。2.文本向量+原始文本块存入本地内存向量库。
print("向量库创建完成")
#4.检索器
retriever=vector_store.as_retriever(search_kwargs={"k":3})
#将向量库转为检索器对象,提供统一检索接口 k=3:相似度检索式,取出和用户问题语义最接近的三个文本块作为参考上下文
#5.RAG链
prompt=PromptTemplate.from_template(
"""请根据以下参考资料回答用户的问题。参考资料:{context} 用户问题:{question} 
要求:基于参考资料回答,如果资料中没有相关信息,请明确说未找到相关信息。
答案:""")
def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])
rag_chain=({
    "context":lambda q:format_docs(retriever.invoke(q)), #retriever.invoke(q):用用户问题去向量库检索相关的文档块
    "question":lambda q:q
}|prompt|llm|StrOutputParser())
#6.测试回答
questions=["什么是Ai agent","Ai agent英文解释","今天啥天气"]
for q in questions:
    print(f"\n问题:{q}");
    answer=rag_chain.invoke(q);
    print(f"答案:{answer}")

分割成2个文档块


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

嵌入模型加载完成
向量库创建完成

问题:什么是Ai agent
答案:AI Agent（人工智能代理/智能体）是一个能够感知环境、进行决策并执行行动，以达成特定目标的智能软件实体。其本质是自动执行任务的程序，核心在于让模型不只回答问题，而是按步骤完成动作。它不仅仅是回答问题的聊天机器人，更是能够动手做事的智能执行者。

问题:Ai agent英文解释
答案:AI Agent 的英文全称是 Artificial Intelligence Agent。

问题:今天啥天气
答案:未找到相关信息


In [1]:
import os
from datetime import datetime
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

@tool
def getcurrent_time() -> str:
    """获取当前日期和时间。当用户询问时间、日期、今天是几号时调用工具"""
    return datetime.now().strftime("%Y年%m月%d日 %H:%M:%S")#y,m,d year,month,day

@tool
def calculator(expression: str) -> str:
    """执行简单的数学计算。当用户需要计算时调用此工具。
    输入：数学表达式，如"2+3"或"5*4" """
    try:
        allowed = set("0123456789+-*/(). ")
        if not all(c in allowed for c in expression):
            return "错误：表达式包含不允许的字符"
        result = eval(expression)
        return f"计算结果：{expression} = {result}"
    except Exception as e:
        return f"计算错误：{str(e)}"
    
@tool
def weather(query: str) -> str:
    """获取指定时间或地点的天气预报。当用户询问天气时调用此工具。
    输入：表示时间或地点的查询词，如"明天"、"西安"、"周末天气"等
    """
    return f"天气预报：{query}，预计晴转多云，温度 22-28°C,微风。"
tools = [getcurrent_time, calculator,weather]
print(f"已定义 {len(tools)} 个工具")

llm = ChatOpenAI(
    api_key=os.environ.get("AGNES_API_KEY"),
    base_url="https://apihub.agnes-ai.com/v1",
    model="agnes-2.0-flash",
    temperature=0
)
"""回答按照下列格式要求：
  Thought: 我需要做什么
  Action: 工具名称
  Action Input: 工具的输入参数
  Thought: 我已经知道答案
  Final Answer: 最终答案"""
# 创建 agent（system_prompt 必须是字符串，不是模板对象）
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
你是一个智能助手，必须优先调用工具来回答问题。
- 调用工具后，必须直接使用工具返回的结果作为最终答案，不要修改或补充。
- 不要自己猜测时间或日期。
    """#这里直接限制agent输出格式
)

# 测试
questions = ["现在是什么时间？顺便计算 100+200","明天西安天气?"]

for q in questions:
    print(f"\n{'='*40}\n问题: {q}")
    try:
        response = agent.invoke({"messages": [{"role": "user", "content": q}]})
        #Agent会根据自己的内部逻辑(工具调用+LLM推理)生成回答,并返回完整的结果对象。
        final_message = response["messages"][-1]#responsemessages是本次调用的完整对话历史,-1取agent最后回答
        print(f"最终答案: {final_message.content}")#这里是取回答
    except Exception as e:
        print(f"错误: {e}")

已定义 3 个工具

问题: 现在是什么时间？顺便计算 100+200
最终答案: 现在是2026年06月21日 12:10:04。  
100+200 的计算结果是 300。

问题: 明天西安天气?
最终答案: 明天西安天气预计晴转多云，温度在22-28°C之间，风力为微风。


In [ ]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
#配置llm
llm=ChatOpenAI(api_key=os.environ.get("AGNES_API_KEY"),base_url="https://apihub.agnes-ai.com/v1",model="agnes-2.0-flash",temperature=0)
#1.文档内容
sample_doc="""AI Agent（人工智能体）是一种能够自主感知环境、做出决策并执行动作的智能系统。
它通常由大语言模型（LLM）驱动，具备规划、记忆、工具调用等能力。典型应用包括：自动化客服、个人助理、代码生成、数据分析等。
某科技公司正在研发基于多智能体协作的自动化办公系统。教授张伟研究了情感计算,Ai agent等课题。"""
with open("test_doc.txt","w",encoding="utf-8") as f:
    f.write(sample_doc) #创建一个名为testdoc.txt的文本文件,写入样例知识内容

#2.加载和分割文档
documents=TextLoader("test_doc.txt",encoding="utf-8").load()
text_splitter=RecursiveCharacterTextSplitter(chunk_size=150,chunk_overlap=20)
chunks=text_splitter.split_documents(documents)
print(len(chunks))

#3.向量检索
print("加载嵌入模型")
embeddings=HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",model_kwargs={"device": "cpu"})
vectorstore=Chroma.from_documents(documents=chunks,embedding=embeddings)#将文档转为向量
retriever=vectorstore.as_retriever(search_kwargs={"k":2})#每次检索返回最相关的两个文档块

def retrieve(query):
    docs=retriever.invoke(query);
    return "\n".join([doc.page_content for doc in docs])
#4.提示词
prompt_template=PromptTemplate.from_template("""你是一个知识助手.
[对话历史]{history} [当前问题]{question} [参考资料]{context}
要求:
1.如果当前问题和对话历史有关,请明确指出与前文的联系。2.回答必须基于材料,不要编造。3.如果上一轮提到了某个主题(如人物、机构),本轮继续讨论它时,请显式重复该主体的身份。
回答:""")
#5.对话记忆存储
chat_history=[]
def rag_with_memory(question):
    context=retrieve(question)
    history_text="\n".join(chat_history[-4:])if chat_history else "无历史对话"#取最近四条对话记录,两轮回答,拼成历史文本
    prompt=prompt_template.format(
        history=history_text,question=question,context=context)
    answer=llm.invoke(prompt).content
    chat_history.append(f"用户:{question}");chat_history.append(f"助手:{answer}")
    return answer
#问题
questions=["AI Agent 是什么?","它有哪些典型应用","张伟教授的研究是否与我们讨论话题相同？他还研究什么课题？"]
print("带记忆的RAG系统")
for q in questions:
    print(f"用户:{q}")
    ans=rag_with_memory(q)
    print(f"助手:{ans}\n")

2
加载嵌入模型


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

带记忆的RAG系统
用户:AI Agent 是什么?
助手:AI Agent（人工智能体）是一种能够自主感知环境、做出决策并执行动作的智能系统。它通常由大语言模型（LLM）驱动，具备规划、记忆、工具调用等能力。典型应用包括：自动化客服、个人助理、代码生成、数据分析等。

用户:它有哪些典型应用
助手:承接上文关于“AI Agent 是什么”的讨论，结合提供的参考资料，AI Agent 的典型应用及研究方向包括：

1. **通用典型应用**：根据前文定义，AI Agent 的典型应用包括自动化客服、个人助理、代码生成和数据分析。
2. **自动化办公系统**：某科技公司正在研发基于多智能体协作的自动化办公系统。
3. **科研与特定领域应用**：教授张伟（其研究课题包括情感计算和 AI Agent）的研究团队专注于提升 Agent 在多步推理中的准确性和稳定性，并涉及情感计算方向。

用户:张伟教授的研究是否与我们讨论话题相同？他还研究什么课题？
助手:承接上文关于“AI Agent 是什么”及其典型应用的讨论，当前问题聚焦于具体人物张伟教授的研究领域。

1. **与前文的联系**：在上一轮回答中，我提到“教授张伟（其研究课题包括情感计算和 AI Agent）的研究团队专注于提升 Agent 在多步推理中的准确性和稳定性”。本轮问题正是基于这一提及的人物展开，进一步确认其研究内容与之前讨论的“AI Agent”主题的一致性，并询问其其他研究课题。

2. **张伟教授的研究是否与讨论话题相同**：是的，相同。根据参考资料，教授张伟研究的课题包括 **AI Agent**，这与我们当前讨论的核心话题完全一致。

3. **他还研究什么课题**：除了 AI Agent 之外，教授张伟还研究 **情感计算**。

综上所述，教授张伟是研究 **AI Agent** 和 **情感计算** 的学者，其研究方向涵盖了我们正在讨论的 AI Agent 领域。

